In [2]:
from gam_rs_utils.utils import *
from src.prepare_gam import get_fastsparse
from src.rset_opt import RSetOPT
from src.run_app import get_models_from_rset
from method_scripts.results import Results

In [17]:
dn = 'bank'
data = pd.read_csv(f'datasets/{dn}.csv')
l0 = 0.001
l2 = 0.001
m = 1.01

# data is 17 features
# w becomes 3719 features

# feature: house
# 3  7 12
# 0  0  1 0 0
# 0  0  0 0 1

# 3,4,5,6,7
# 1 1 1 0 0
# 1 1 1 1 1

fastsparse_data = Results.create_fastsparse_dataset(dn, l0, l2)
cum_X = fastsparse_data['X']
y = fastsparse_data['y']
header = fastsparse_data['header']
w = fastsparse_data['w']

bin_X = DatasetUtils.convert_cumulative_to_binned(cum_X.values, header[1:])[0]
bin_X = np.hstack((np.ones((bin_X.shape[0],1)), bin_X))

sparse_X, sparse_header = utils.binary_to_one_hot(data.iloc[:,:-1], w, header)

print("data shape", data.shape)
print("w", len(w), "y", len(y), "header", len(header))
print("X_orig shape", cum_X.shape)
print("X_new.shape", sparse_X.shape, "sparse_header", sparse_header)

Loading cached fastsparse dataset for bank with l0=0.001 and l2=0.001
data shape (4521, 17)
w 3719 y 4521 header 3719
X_orig shape (4521, 3718)
X_new.shape (4521, 21) sparse_header ['intercept', 'housing<=0.0', '0.0<housing<=1', 'loan<=0.0', '0.0<loan<=1', 'contact<=1.0', '1.0<contact<=2', 'month<=9.0', '9.0<month<=11', 'duration<=211.0', '211.0<duration<=348.0', '348.0<duration<=645.0', '645.0<duration<=770.0', '770.0<duration<=3025', 'pdays<=374.0', '374.0<pdays<=871', 'previous<=0.0', '0.0<previous<=1.0', '1.0<previous<=25', 'poutcome<=1.0', '1.0<poutcome<=3']


In [5]:
# start = time()

sparse_gam_file = prepare_sparse_gam(dn, l0, l2, m, sparse_X, y, header, sparse_header)

model = RSetOPT(sparse_gam_file)
model.finetune_ellipsoid()
H_opt = model.get_normalized_H()
model.update_file(H_opt, model.w_orig)

# end = time()

with open(sparse_gam_file, 'rb') as f:
    sparse_gam_data = pickle.load(f)

sparse_X = sparse_gam_data["X"]
y = sparse_gam_data["y"]
sparse_header = sparse_gam_data["header_new"]
sample_p = sparse_gam_data["sample_proportion"]

print('X shape', sparse_X.shape)
print('y shape', y.shape)
print('header shape', len(header))
print('sample_p shape', sample_p.shape)

objective: 0.2443484497913396 objective in LR 0.2443484497913396
m:1.01, log objective:0.2443484497913396, eps:0.246791934289253
----------- before optimization -----------
volume proportional to  tensor(3.5345e+24, dtype=torch.float64, grad_fn=<MulBackward0>)
----------- after optimization -----------
volume proportional to  tensor(1.4767e+24, dtype=torch.float64, grad_fn=<MulBackward0>)
X shape (4521, 21)
y shape (4521,)
header shape 3719
sample_p shape (21,)


In [6]:
n_samples = 100
sampling = 'uniform'
distance_metric = None
r_min = None

w_samples, rset = get_models_from_rset(
    sparse_gam_file, n_samples=n_samples, plot_shape=False, 
    sampling=sampling, distance_metric=distance_metric, r_min=r_min,
)

print('w_samples shape', w_samples.shape)

w_samples shape (100, 21)


In [ ]:
# w_samples_zeroed = ModelUtils.hard_threshold_samples(w_samples, rset, 17)
# ModelUtils.print_results_summary(w_samples_zeroed, sparse_gam_data['w_opt'], X, y, l2, sample_p, end - start)

array([-2.04192107, -2.44651175, -3.86067603, ..., -3.22057456,
       -2.72400194, -2.44651175])

In [ ]:
def get_header_object(header):
    header_object = defaultdict(list)
    header_object['intercept']
    for h in header[1:]:
        feature = re.search(r'([a-zA-Z]+)', h).group(1)
        threshold = [float(t) for t in re.findall(r'[\d.]+', h)][-1]
        header_object[feature].append(threshold)
    return header_object

def get_expanded_weights(weights, sparse_header, header):
    new_w = [weights[0]]
    wi = 1
    for h, thresholds in header.items():
        if h == 'intercept':
            continue
        if h not in sparse_header:
            new_w.extend([0.0] * len(thresholds))
            continue
        sparse_thresholds = sparse_header[h]
        si = 0
        for t in thresholds:
            if t > sparse_thresholds[si]:
                wi += 1
                si += 1
            new_w.append(weights[wi])
        wi += 1
    return np.array(new_w)

header_object = get_header_object(header)
sparse_header_object = get_header_object(sparse_header)

In [ ]:
for i in range(len(w_samples)):
    expanded_w = get_expanded_weights(w_samples[i], sparse_header_object, header_object)
    a = ModelUtils.get_logits(sparse_X, w_samples[i])
    b = ModelUtils.get_logits(bin_X, expanded_w)
    if not np.allclose(a,b):
        print(f'w_samples[{i}] is not equal to expanded_w')
        print(a - b)
    
